# 01 — OCR Engine Evaluation: Tesseract vs EasyOCR vs docTR

This notebook benchmarks three open-source OCR engines on a stratified sample
of the raw PDF dataset. For each sampled page it measures:

| Metric | Method |
|--------|--------|
| **Processing time** | wall-clock seconds per page |
| **Word count** | proxy for extraction coverage |
| **Confidence** | Tesseract word-level mean; EasyOCR/docTR detection score |
| **CER** (Character Error Rate) | only for PDFs with an embedded text layer (used as reference) |

Results are saved incrementally to `../../data/ocr_eval/results.csv`
so the notebook can be interrupted and resumed safely.

> **Note:** Asset creation (page images + OCR text files) is handled by the
> CLI pipeline — see README for instructions. Run all three engines first:
> ```bash
> python src/assets/run.py --engine tesseract --supervise
> python src/assets/run.py --engine easyocr --supervise
> python src/assets/run.py --engine doctr --supervise
> ```

## Setup

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path("../src/assets").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Source path:", SRC_DIR)

In [ ]:
import csv
import random
from collections import defaultdict
from pathlib import Path

import fitz
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from services.pdf_loader import PdfLoader
from services.tesseract_ocr import TesseractOcr
from services.easyocr_ocr import EasyOcrEngine
from services.doctr_ocr import DocTROcrEngine

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})
print("Imports OK")

## Configuration

In [ ]:
RAW_DATA_PATH = "../../data/raw_data"
ASSETS_PATH   = Path("../../data/assets")
EVAL_OUT_DIR  = Path("../../data/ocr_eval")
CHART_DIR     = EVAL_OUT_DIR / "charts"

EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)

DOCS_PER_TYPE = 3
PAGES_PER_DOC = 3
IMAGE_DPI     = 200
RANDOM_SEED   = 42

EASYOCR_GPU = True
DOCTR_GPU   = True

print(f"Docs per type : {DOCS_PER_TYPE}")
print(f"Pages per doc : {PAGES_PER_DOC}")
print(f"Output        : {EVAL_OUT_DIR.resolve()}")

## Build stratified sample

In [ ]:
random.seed(RANDOM_SEED)

loader   = PdfLoader(raw_data_path=RAW_DATA_PATH)
all_docs = loader.get_all_documents()

by_type = defaultdict(list)
for d in all_docs:
    by_type[d.doc_type].append(d)

sample_docs = []
for dtype, docs in sorted(by_type.items()):
    chosen = random.sample(docs, min(DOCS_PER_TYPE, len(docs)))
    sample_docs.extend(chosen)
    print(f"  {dtype:<30} sampled {len(chosen)}/{len(docs)}")

print(f"\nTotal sample: {len(sample_docs)} documents")

## Helper functions

**CER (Character Error Rate)** is computed only when the PDF has an embedded
text layer. PyMuPDF's `get_text()` output serves as the reference transcript.

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute Levenshtein edit distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if not s2:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]


def compute_cer(reference: str, hypothesis: str) -> float:
    """Character Error Rate. Returns -1.0 if reference is empty."""
    ref = reference.strip()
    hyp = hypothesis.strip()
    if not ref:
        return -1.0
    return round(levenshtein_distance(ref, hyp) / len(ref), 4)


def pdf_embedded_text(pdf_path: str, page_idx: int) -> str:
    """Extract embedded text from a PDF page (empty string for pure scans)."""
    doc = fitz.open(pdf_path)
    try:
        return doc[page_idx].get_text("text").strip()
    finally:
        doc.close()


def render_page(pdf_path: str, page_idx: int, dpi: int = 200) -> Image.Image:
    """Render a single PDF page to a PIL Image."""
    doc = fitz.open(pdf_path)
    try:
        pix = doc[page_idx].get_pixmap(dpi=dpi)
        return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    finally:
        doc.close()

print("Helpers defined")

## Initialise OCR engines

In [ ]:
tesseract = TesseractOcr(lang="eng", dpi=IMAGE_DPI)
easyocr_e = EasyOcrEngine(langs=["en"], gpu=EASYOCR_GPU)
doctr_e   = DocTROcrEngine(gpu=DOCTR_GPU)

engines = [tesseract, easyocr_e, doctr_e]
print(f"Engines: {[e.name for e in engines]}")

## Run benchmark

Results are appended to `results.csv` after each page — safe to interrupt and resume.

In [ ]:
RESULTS_CSV = EVAL_OUT_DIR / "results.csv"
FIELDNAMES  = [
    "doc_type", "doc_name", "page_idx",
    "engine", "elapsed_s", "word_count", "confidence", "cer",
    "has_embedded_text",
]

done_set = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    for _, row in prev.iterrows():
        done_set.add((row["doc_name"], int(row["page_idx"]), row["engine"]))
    print(f"Resuming — {len(done_set)} measurements already recorded")

write_header = not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0
csv_file = open(RESULTS_CSV, "a", newline="", encoding="utf-8")
writer   = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
if write_header:
    writer.writeheader()

try:
    for doc in tqdm(sample_docs, desc="Documents"):
        n_pages = min(doc.page_count, PAGES_PER_DOC)

        for page_idx in range(n_pages):
            embedded_text = pdf_embedded_text(doc.absolute_filepath, page_idx)
            has_embedded  = len(embedded_text) > 50
            image         = render_page(doc.absolute_filepath, page_idx, IMAGE_DPI)

            for engine in engines:
                key = (doc.doc_name, page_idx, engine.name)
                if key in done_set:
                    continue

                result = engine.extract_page(image)
                cer    = compute_cer(embedded_text, result.text) if has_embedded else -1.0

                writer.writerow({
                    "doc_type":          doc.doc_type,
                    "doc_name":          doc.doc_name,
                    "page_idx":          page_idx,
                    "engine":            engine.name,
                    "elapsed_s":         round(result.elapsed_seconds, 4),
                    "word_count":        result.word_count,
                    "confidence":        result.confidence,
                    "cer":               cer,
                    "has_embedded_text": has_embedded,
                })
                csv_file.flush()
                done_set.add(key)
finally:
    csv_file.close()

print(f"\nResults saved to: {RESULTS_CSV}")

## Load results

In [ ]:
df = pd.read_csv(RESULTS_CSV)
print(df.shape)
df.head()

## Summary statistics

In [ ]:
summary = (
    df.groupby("engine")[["elapsed_s", "word_count", "confidence"]]
    .agg(["mean", "median", "std"])
    .round(4)
)

cer_summary = (
    df[df["has_embedded_text"] & (df["cer"] >= 0)]
    .groupby("engine")["cer"]
    .agg(["mean", "median", "std"])
    .round(4)
    .rename(columns=lambda c: f"cer_{c}")
)

print("=== Core metrics ===")
display(summary)
print("\n=== CER (pages with embedded text only) ===")
display(cer_summary)

## Chart 1 — Processing time distribution

In [ ]:
PALETTE = {
    "tesseract": "#2563EB",
    "easyocr":   "#0F766E",
    "doctr":     "#EA580C",
}

engines_ordered = [e for e in ["tesseract", "easyocr", "doctr"] if e in df["engine"].unique()]
data_by_engine  = [df[df["engine"] == e]["elapsed_s"].values for e in engines_ordered]
colors          = [PALETTE[e] for e in engines_ordered]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(
    data_by_engine,
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2),
    widths=0.45,
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

rng = np.random.default_rng(42)
for i, (data, color) in enumerate(zip(data_by_engine, colors), start=1):
    jitter = rng.uniform(-0.12, 0.12, size=len(data))
    ax.scatter(i + jitter, data, alpha=0.35, s=14, color=color, zorder=3)

ax.set_xticks(range(1, len(engines_ordered) + 1))
ax.set_xticklabels(engines_ordered)
ax.set_ylabel("Processing time [s / page]")
ax.set_title("OCR Processing Time per Page")
fig.tight_layout()
fig.savefig(CHART_DIR / "01_time_distribution.png", dpi=300)
plt.show()

## Chart 2 — Mean processing time by document type

In [ ]:
pivot_time = (
    df.groupby(["doc_type", "engine"])["elapsed_s"]
    .mean()
    .unstack("engine")
    .sort_values(by="tesseract", ascending=True)
)

n_engines = len(pivot_time.columns)
width = 0.7 / n_engines

fig, ax = plt.subplots(figsize=(11, max(6, len(pivot_time) * 0.5 + 1.5)))
x = np.arange(len(pivot_time))

for i, engine in enumerate(pivot_time.columns):
    offset = (i - n_engines / 2 + 0.5) * width
    ax.barh(x + offset, pivot_time[engine], height=width,
            label=engine, color=PALETTE.get(engine, "grey"))

ax.set_yticks(x)
ax.set_yticklabels(pivot_time.index)
ax.set_xlabel("Mean time [s / page]")
ax.set_title("Mean OCR Processing Time by Document Type")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_DIR / "02_time_by_type.png", dpi=300)
plt.show()

## Chart 3 — Confidence score distribution

In [ ]:
conf_df = df[df["confidence"] >= 0]
engines_with_conf = [e for e in ["tesseract", "easyocr", "doctr"] if e in conf_df["engine"].unique()]
n = len(engines_with_conf)

fig, axes = plt.subplots(n, 1, figsize=(10, 4 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, engine in zip(axes, engines_with_conf):
    data = conf_df[conf_df["engine"] == engine]["confidence"]
    color = PALETTE.get(engine, "grey")
    ax.hist(data, bins=20, color=color, alpha=0.85,
            edgecolor="white", linewidth=0.4)
    ax.set_title(engine, fontsize=14, fontweight="bold", color=color)
    ax.set_ylabel("Number of pages")
    ax.set_xlim(0, 1)

axes[-1].set_xlabel("Confidence (0-1)")
fig.suptitle("OCR Confidence Score Distribution", fontsize=16, y=1.01)
fig.tight_layout()
fig.savefig(CHART_DIR / "03_confidence.png", dpi=300, bbox_inches="tight")
plt.show()

## Chart 4 — CER by document type (pages with embedded text only)

In [ ]:
cer_df = df[(df["has_embedded_text"]) & (df["cer"] >= 0)]

if cer_df.empty:
    print("No pages with embedded text found — CER chart skipped.")
else:
    first_engine = cer_df["engine"].iloc[0]
    pivot_cer = (
        cer_df.groupby(["doc_type", "engine"])["cer"]
        .mean()
        .unstack("engine")
        .sort_values(by=first_engine, ascending=False)
    )

    n_engines = len(pivot_cer.columns)
    width = 0.7 / n_engines

    fig, ax = plt.subplots(figsize=(11, max(6, len(pivot_cer) * 0.5 + 1.5)))
    x = np.arange(len(pivot_cer))

    for i, engine in enumerate(pivot_cer.columns):
        offset = (i - n_engines / 2 + 0.5) * width
        ax.barh(x + offset, pivot_cer[engine], height=width,
                label=engine, color=PALETTE.get(engine, "grey"))

    ax.set_yticks(x)
    ax.set_yticklabels(pivot_cer.index)
    ax.set_xlabel("Mean CER (lower is better)")
    ax.set_title("Character Error Rate by Document Type\n(pages with embedded text layer)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(CHART_DIR / "04_cer_by_type.png", dpi=300)
    plt.show()

## Chart 5 — Speed vs coverage scatter

In [ ]:
scatter_df = (
    df.groupby(["doc_type", "engine"])
    .agg(mean_time=("elapsed_s", "mean"), mean_words=("word_count", "mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 7))
for engine, grp in scatter_df.groupby("engine"):
    ax.scatter(grp["mean_time"], grp["mean_words"], label=engine,
               s=80, alpha=0.85, color=PALETTE.get(engine, "grey"))
    for _, row in grp.iterrows():
        ax.annotate(row["doc_type"], (row["mean_time"], row["mean_words"]),
                    fontsize=9, ha="left", va="bottom", alpha=0.75)

ax.set_xlabel("Mean time [s / page]")
ax.set_ylabel("Mean word count")
ax.set_title("Speed vs Coverage by Document Type")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_DIR / "05_speed_vs_words.png", dpi=300)
plt.show()